# kube-apiserver Attack Reproduction
# parlez su script ajouté
## 1. Mechanism overview

This section details the mechanism described in the paper and the figures it reports, so they can be directly compared to our own results in section 3.

### 1.1 The default service account token as an attack surface

The paper describes a set of kube-apiserver attack strategies exploiting the default service account token, automatically mounted inside every pod at `/var/run/secrets/kubernetes.io/serviceaccount/token` (§3.5). RBAC restricts which interfaces this token can reach, but not how often it can be used to reach them. Table 4 lists a fixed set of 14 interfaces reachable with the default token regardless of the pod's own RBAC configuration, including `/healthz`, `/livez`, `/readyz`, `/version`, `/api`, `/apis`, `/openapi`, and the selfsubjectreview/selfsubjectaccessreview/selfsubjectrulesreview endpoints. Because none of these interfaces are rate limited, any pod in the cluster, however unprivileged, can send an unbounded volume of authenticated requests against them.

The strategy reproduced here corresponds to the paper's Case 2 (§4.3), where a small number of malicious containers hold open many connections against one of these allowed interfaces at once.

### 1.2 Two consequences of the same connection flood

The paper describes the attack as producing two effects at once:

1. System-level saturation via `nf_conntrack`. Sending a large number of TCP connections to any allowed interface fills the kernel's connection tracking table quickly. This effect depends only on the volume and persistence of connections reaching the apiserver, not on which interface or what content is requested: a conntrack entry is created as soon as the TCP handshake completes at the kernel level, independently of whether the TLS handshake or the HTTP request above it ever finishes.

2. CPU amplification via the specific interface queried. The interface chosen still matters for a second effect: `/healthz`, for instance, makes the kube-apiserver query other control-plane components internally as part of answering the health check (for instance etcd's own health key), so a flood of `/healthz` requests does not only stress the apiserver itself but propagates CPU load onto the components it queries in turn.

### 1.3 What the paper reports as impact

For the kube-apiserver attack (Case 2) in the local environment, the paper reports that 5 malicious containers fill the nf_conntrack table within 5 minutes, that the victim container suffers 30% packet loss when connecting to the control plane, and that the kube-apiserver together with the other control-plane components it queries consume 90% additional CPU (§4.3). These are the reference figures our own measurements are compared against in section 3.

## 2. Reproduction: manifests and methodology

### 2.1 Attack topology

The attack reuses the namespace and node isolation described in the environment section: the attacker pods run in `attacker-ns`, pinned to `k8s-w1` via `nodeName`, while the victim pod (`safe-pod`) runs in `safe-ns`, pinned to `k8s-w2`. Both namespaces enforce the `restricted` Pod Security Standard, so the malicious containers operate under the same constraints assumed by the paper's threat model.

### 2.2 Final manifest

The attack deployment (`apiserver-flood.yaml`, `replicas: 5`, matching the paper's Case 2 setup) runs a Python asyncio program in each attacker pod. Each pod opens a configurable number of TLS connections (`CONNS`) directly to the kube-apiserver, authenticated with its own default service account token, and holds them open rather than opening and closing them in a loop: once a connection is established it is kept alive with a periodic `GET /healthz` request, so that each held connection occupies exactly one conntrack entry for as long as it survives, and the interface queried also produces the CPU amplification effect described in section 1.2.

Two implementation details matter for reaching a sustained saturation rather than a brief spike:

- The apiserver is targeted by its ClusterIP (`10.96.0.1:443`, from `kubectl get svc kubernetes`) rather than the hostname `kubernetes.default.svc`. Connecting by hostname triggers a fresh DNS lookup on every new connection attempt, which under load generated far more conntrack entries (UDP, port 53, toward CoreDNS) than the intended TCP flood itself, confounding the measurement of the apiserver's own load.
- Both the periodic re-ping interval (`random.uniform(20, 60)` seconds) and the retry backoff after a connection failure (`random.uniform(1, 6)` seconds) use a randomized delay rather than a fixed one, for the reason detailed in section 2.3.

The full manifest is in the `manifests/` folder.

### 2.3 Why this design, and not an earlier one

**An initial single-source-IP hypothesis, confirmed.** Because all 5 attacker pods run on the same node, Calico's default `natOutgoing` setting masquerades their outbound traffic to the apiserver behind that node's single IP, which would cap the total number of concurrent connections at what a single IP's ephemeral port range allows (roughly 28,000 ports). This matched what was observed: with `natOutgoing` left enabled, the attack could not reach its intended saturation level. `natOutgoing` was disabled (`false`) so each attacker pod keeps its own distinct source IP, and it has been kept disabled since.

**Open/close connection churn versus held-open connections.** The very first version of the attack reused the thread-based pattern already working for the CoreDNS attack: several threads, each opening a fresh connection, sending a request, closing it, and looping, just adapted from a raw UDP socket to a TCP `HTTPSConnection` for the apiserver. This pattern does not transfer from UDP to TCP. A UDP send has no real connection to close, so a closed UDP flow's conntrack entry expires quickly (on the order of tens of seconds), and a fast loop of many threads keeps recreating entries about as fast as they expire, which is enough to sustain a high count. A closed TCP connection, on the other hand, does not free its conntrack entry right away: it sits in the kernel's TIME_WAIT state for a fixed duration of around 120 seconds first. This turns the table's size into a balance between how fast new connections are created and this fixed 120-second expiry delay, rather than something the loop's speed alone can push arbitrarily high: past the first 120 seconds, connections start expiring at roughly the same rate new ones are opened, so the count settles into a plateau instead of continuing to climb. This is what motivated the switch to connections that are established once and then kept alive: a held-open connection never enters TIME_WAIT while it survives, so it occupies its conntrack entry directly for as long as the attack runs, without being capped by this expiry delay.

**Fixed intervals versus jittered intervals.** With a fixed re-ping interval and a fixed retry backoff, connections established in the same initial burst tended to re-ping, and later retry, at nearly the same moment, producing a synchronized wave of requests against an apiserver already under load. This caused a pronounced sawtooth pattern: the table would saturate, a shared stress event (CPU spike, packet drops once the table is full) would cause many connections to fail at once, the table would partially drain, and the cycle would repeat once the synchronized retries succeeded again. Randomizing both the re-ping interval and the retry backoff spreads this recovery over time instead of concentrating it, which markedly reduced, though did not entirely eliminate, this oscillation: the retry backoff window (`random.uniform(1, 6)` seconds) was tuned as an explicit middle ground between a tighter window (which still triggered visible collapses) and a wider one (which slowed the refill enough to shorten how long the attack spent near saturation).

**Tuning `CONNS` to reach the ceiling without breaking the measurement pipeline.** With `nf_conntrack_max` fixed at 131,072, the target of roughly 26,214 held connections per pod (131,072 divided across 5 pods) is what the deployment aims to sustain. A `CONNS` value that undershoots this target plateaus well below the ceiling; a value that overshoots it by too much saturates the apiserver so completely that even the measurement tooling itself (a `kubectl exec`-based TCP probe, or `kubectl exec` used for any other purpose) can no longer get a response, which defeats the point of measuring packet loss at all. In practice, the right value is not a single fixed number: it depends on the state of the machine running the attacker pods at a given moment, for instance whether a previous run has left residual load behind, so `CONNS` is adjusted slightly from one run to the next (28,000 per pod as a starting point, lowered when a prior run's effects have not fully settled) rather than pinned to one definitive value.

### 2.4 Measurement scripts

Three scripts drive the measurement, kept separate from the attack manifest itself:

**`launch-attack.sh`** deploys the attack. Unlike CoreDNS, the kube-apiserver attack has no equivalent Prometheus counter that needs resetting between runs (the apiserver's `/metrics` endpoint is not reachable with the default service account token, so it is not part of this attack's measurement at all), so no control-plane component restart is required before launching.

**`capture-metrics-loop-attack2.sh`** polls three independent signals at a fixed interval, for a fixed duration, and writes each sample to a CSV: `nf_conntrack_count`/`nf_conntrack_max` and CPU usage (`%us`/`%sy`), both read locally with no dependency on any other node, and a TCP availability probe of the control plane run from the victim pod (`safe-pod`), classified as `OK`, `TIMEOUT`, or `ERROR`. As with the CoreDNS capture script, only `OK` samples are averaged for the packet-loss figure, since a failed probe is a measurement worth keeping, not a zero-valued reading.

**A methodological limitation of the `kubectl exec`-based TCP probe.** `kubectl exec` is itself an API call routed through the kube-apiserver (client, apiserver, kubelet on the node hosting the pod, container runtime, exec inside the container). During this specific attack, that chain shares the very component under attack, so a `kubectl exec`-based probe conflates two different things: whether the victim pod can reach the control plane over TCP, and whether the control plane can still service an exec request at all. `tcp-latency-test-nsenter.sh` addresses this by running directly on the node hosting the victim pod (via SSH), entering the pod's own network namespace with `nsenter` (preserving its real source IP) and issuing the TCP probe without involving the apiserver at all. It is used to sanity-check the `capture-metrics-loop-attack2.sh` packet-loss figures, and can be re-run standalone at any point during an attack for a probe uncoupled from the apiserver's own health.

## 3. Results

Pending: this section will be filled in once both measurement streams have run for a full before/after cycle over the same attack window and been aggregated. Two separate CSVs feed this section, matching the split described in section 2.4: `nf_conntrack_count`/`nf_conntrack_max` and control-plane CPU usage come from converting the live output of `watch-apiserver-metrics.sh`, run on k8s-cp, into a CSV; TCP packet loss on the victim container comes from `capture-tcp-loss-loop-nsenter.sh`, run separately on k8s-w2, entering the victim pod's own network namespace with nsenter to avoid any dependency on the apiserver. 

In [4]:
from aggregate_averages_apiserver import (
    find_watch_file,
    find_cpu_loop_files,
    find_tcp_loss_files,
    load_watch_rows,
    load_cpu_loop_rows,
    load_tcp_ok_rows,
    COLS,
    CPU_LOOP_COLS,
    TCP_COL,
    LABELS,
)
import pandas as pd
from IPython.display import display, HTML

def load_combined(label):
    watch_files = find_watch_file(label, base_dir="../results")
    cpu_loop_files = find_cpu_loop_files(label, base_dir="../results")
    tcp_files = find_tcp_loss_files(label, base_dir="../results")

    watch = pd.concat([load_watch_rows(f) for f in watch_files], ignore_index=True) if watch_files else pd.DataFrame(columns=COLS)
    cpu_loop = pd.concat([load_cpu_loop_rows(f) for f in cpu_loop_files], ignore_index=True) if cpu_loop_files else pd.DataFrame(columns=CPU_LOOP_COLS)
    tcp = pd.concat([load_tcp_ok_rows(f) for f in tcp_files], ignore_index=True) if tcp_files else pd.DataFrame(columns=[TCP_COL])

    return watch, cpu_loop, tcp

before_watch, before_cpu_loop, before_tcp = load_combined("before")
after_watch, after_cpu_loop, after_tcp = load_combined("after")

rows = COLS + CPU_LOOP_COLS + [TCP_COL]
summary = pd.DataFrame({
    "Before": pd.concat([before_watch[COLS].mean(), before_cpu_loop[CPU_LOOP_COLS].mean(), before_tcp[[TCP_COL]].mean()]),
    "After": pd.concat([after_watch[COLS].mean(), after_cpu_loop[CPU_LOOP_COLS].mean(), after_tcp[[TCP_COL]].mean()]),
}).round(1).reindex(rows).rename(index=LABELS)

display(HTML("<h2>Metrics comparison before / after attack</h2>"))
display(summary.style.set_properties(**{"font-size": "16pt"}).set_table_styles(
    [{"selector": "th", "props": [("font-size", "16pt")]}]
))

,Before,After
nf_conntrack_count,328.500000,128534.700000
"CPU user, whole node (%)",3.700000,5.100000
"CPU system, whole node (%)",3.400000,6.100000
CPU kube-apiserver process (%),8.700000,33.400000
CPU etcd process (%),4.100000,2.300000
CPU kube-apiserver + etcd (%),12.800000,35.700000
TCP packet loss to control plane (%),0.000000,47.300000


## 3. Results

| Metric | Paper, local env. (§4.3) | Our reproduction |
|---|---|---|
| Time to fill nf_conntrack | ~5 minutes (5 containers) | not precisely measured, but occupancy rose from 288/131,072 to full saturation in about 106 seconds in one capture, faster than the paper's local figure |
| Packet loss on the victim container | 30% | mean 47.3% (`tcp_probe_status == OK` samples only), with individual samples reaching up to 100% |
| CPU consumed by kube-apiserver | 90% additional (attributed jointly to kube-apiserver and the components it queries) | 8.7% to 33.5%, a 24.8 point increase |
| CPU consumed by etcd | included in the 90% figure above | 4.1% to 2.3%, a 1.9 point decrease |

Four things stand out from this comparison.

**nf_conntrack saturation is confirmed, and reached faster than in the paper's local reproduction.** The paper reports the table filling up within 5 minutes with 5 containers in its local environment. We did not measure the exact fill time with a dedicated instrument, but our own samples show it is considerably faster here: in one capture, occupancy rose from 288/131,072 at 13:10:03 to full saturation (131,072/131,072) by 13:11:49, about 106 seconds later. Once reached, saturation is sustained for nearly the whole attack: a 98.1% mean occupancy, with most individual samples sitting at exactly 131,072/131,072.

**Packet loss on the victim container is higher than the paper's local figure.** The paper reports 30% packet loss in its local environment; our mean is 47.3% (`tcp_probe_status == OK` samples only), with individual sampling windows recording up to 100% loss (all 20 probe attempts failing) at moments closest to full saturation. This spread makes sense given the paper's own description of the effect: once the table saturates, packets are dropped randomly, and how often depends on how full the table is at that instant. Our capture shows exactly this kind of variation: occupancy oscillates during the attack, with partial drains down to around 117,000 to 120,000 alternating with moments at the full 131,072 ceiling (section 2.3), so the loss rate rises and falls along with it, higher during full saturation, lower during the partial drains. The 47.3% mean is computed over the whole window and blends both kinds of moments together, and still comes out above the paper's local figure.

**kube-apiserver CPU consumption is measured against the process directly, not the whole node.** The paper attributes its 90% figure jointly to kube-apiserver and the components it queries, not to whole-node CPU, so `cpu_kube_apiserver` and `cpu_etcd` (read from `/proc/<pid>/stat`, section 2.4) are compared against it instead of the node-wide `top` reading used earlier in this pipeline. `kube-apiserver` rises from 8.7% to 33.5%, a 24.8 point increase, below the paper's combined 90% figure. As with the CoreDNS attack, the paper does not specify whether its own CPU figures are capped at 100% (normalized across all cores) or can exceed it like ours, so the two numbers may not sit on the same scale.

**etcd CPU consumption decreases during the attack, contrary to the paper's mechanism.** Where the paper attributes part of its 90% figure to etcd (queried by kube-apiserver via `/healthz`, section 1.2), our measurement shows `etcd` CPU falling from 4.1% to 2.3%, a 1.9 point decrease, reproducible across separate before/after runs. `/healthz` triggers only a lightweight health read on etcd, not a write, which may explain why it adds little load on its own. One possible explanation for the decrease, not confirmed by this data, is that kube-apiserver's own saturation under attack could delay or drop legitimate cluster traffic that normally reaches etcd (kubelet heartbeats, controller watches), lowering etcd's baseline load below its at-rest level. This is reported as observed, without a confirmed explanation for the decrease.